In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)
from langchain_chroma import Chroma
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
## Index
loader = PyPDFLoader("os-sertoes.pdf")
doc = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(doc)

In [ ]:
## Retriever
vectorstore = Chroma(persist_directory="db", embedding_function=OpenAIEmbeddings())
vectorstore.add_documents(splits)

base_retriever = vectorstore.as_retriever()

In [ ]:
## Compression
compressor = CohereRerank(top_n=5)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [ ]:
## Generation
llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}
    Question: {question}"""
)

In [ ]:
## Chain
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


rag_chain = (
        {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)